# 02h — P4, asymmetric transfer

Registered prediction. Estimated **~25 min** on a T4. Suggested kernel: `emocap-p4`.

**P4:** *"Models trained on S score lower under an H-trained judge than H-trained models do
under an S-trained judge (asymmetric transfer)."* Falsified if symmetric, or reversed.

## Why this is a notebook and not a laptop job

P4 is about **judges**, not generators — no arm is retrained. But it does train two
DistilRoBERTa classifiers and cross-validate each five times, which is about an hour of
sustained GPU. The first attempt ran locally, wedged on MPS overnight in uninterruptible
wait, and threw away a completed cross-validation. A T4 does the same work in roughly 25
minutes and cannot be put to sleep mid-run.

## What it measures, and why a raw number would not do

If the H-judge scores S captions at 0.30, that is uninterpretable on its own: it could mean
transfer fails, or it could mean the H-judge is a weak classifier. Only each judge's
accuracy **on its own corpus** separates those, so that is measured first by 5-fold CV, and
transfer is reported as a **drop from own-corpus accuracy**. An asymmetry claim needs the
two drops to differ, not the raw numbers.

Two levels, because P4's wording and its mechanism are not the same thing:

| level | what it uses | what it answers |
|---|---|---|
| corpus | each judge on the other's **reference** captions | pure judge asymmetry — the control. If the corpora already transfer asymmetrically, anything measured on generated text inherits it |
| generated | each judge on the arms' **generated** captions | what P4 literally says; the reported test |

The P4 pair is `S_unpaired` vs `H_unpaired` — matched at 4,390 cells and one caption per
image, the only pair where "trained on S" and "trained on H" differ by provenance alone.
`S_paired25` would confound provenance with 46× the gradient steps. Other arms are reported
for context and carry no verdict.

**No magnitude was registered for P4** — only a direction. The script reports the direction
and the size and thresholds neither.

## Setup

**Attach two datasets:**

- `emocap-v2-arms` — the reference captions the judges train on
- `emocap-v2-predictions` — the arms' generated captions, **references stripped**

That second dataset contains only captions this project's own models wrote. No Flickr8k and
no Personality-Captions text is in it, which is a better answer to the redistribution
question than relying on the dataset being private.

**Accelerator:** GPU **T4 x2**. **Internet:** on (the judges start from `distilroberta-base`).

> P100 will not work — sm_60 against a PyTorch build shipping sm_70 and up.

Pull the result — it is one small JSON:

    kaggle kernels output <owner>/emocap-p4 -p tmp/emocap-p4 \
        --page-size 200 --file-pattern 'p4_'


In [ ]:
# ── parameters ────────────────────────────────────────────────────────────
DATA = "/kaggle/input/emocap-v2-arms"
PREDS = "/kaggle/input/emocap-v2-predictions/predictions"
OUT = "/kaggle/working/results"

# The three 1-caption-per-image arms are scored by default. The big paired arms cost more
# inference and carry no P4 verdict, so they are opt-in.
INCLUDE_PAIRED = False


In [ ]:
# Clone the EXACT commit the data was built from, pinned to the PREDICTIONS dataset.
#
# Not the arms dataset. Two inputs are attached and they move at different rates: the arms
# have been stable for days, while the prediction export is rebuilt whenever an arm is
# re-decoded and is the input whose contents depend on recent code. Pinning to the arms
# dataset checked out a commit from before scripts/p4_transfer.py existed, and the failure
# was a "no such file" from a subprocess several cells later.
#
# Both commits are printed so a mismatch is visible rather than silent.
import json, os, subprocess, sys
from pathlib import Path

PROV = json.loads(Path(PREDS).parent.joinpath("provenance.json").read_text())
COMMIT = PROV["git_commit"]
arms_commit = json.loads(Path(DATA, "provenance.json").read_text())["git_commit"]
print("predictions built at", COMMIT[:12], "  <- code pinned here")
print("arms built at       ", arms_commit[:12],
      "" if arms_commit == COMMIT else "  (older, fine -- the arms have not changed)")

if not Path("/kaggle/working/EmoCap").exists():
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/asjad2401/EmoCap.git",
                    "/kaggle/working/EmoCap"], check=True)
subprocess.run(["git", "-C", "/kaggle/working/EmoCap", "checkout", "-q", COMMIT], check=True)

# The script this notebook exists to run must be present at that commit -- checked here,
# not discovered by a subprocess failing three cells down.
script = Path("/kaggle/working/EmoCap/scripts/p4_transfer.py")
if not script.exists():
    raise SystemExit(f"{script} is absent at {COMMIT[:12]}. The predictions dataset was "
                     f"pushed from a commit that predates it; re-push it.")
print("code at", COMMIT[:12], "and p4_transfer.py is present")


In [ ]:
sys.path.insert(0, "/kaggle/working/EmoCap/src")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Fail here rather than after the judges are trained.
for name, p in (("arms", Path(DATA, "arms", "S_unpaired.jsonl")),
                ("arms", Path(DATA, "arms", "H_unpaired.jsonl")),
                ("predictions", Path(PREDS, "S_unpaired-f0", "predictions.jsonl"))):
    if not p.exists():
        raise SystemExit(f"missing {name}: {p}\n"
                         "Attach BOTH emocap-v2-arms and emocap-v2-predictions.")
print("both datasets present")

# The prediction export must carry no reference text. Checked against the files actually
# attached, not only where they were built -- this is the property that makes the upload
# free of third-party corpus text, and it is cheap to verify.
first = json.loads(Path(PREDS, "S_unpaired-f0", "predictions.jsonl").open().readline())
if set(first) != {"image_id", "emotion", "generated", "fold"}:
    raise SystemExit(f"unexpected fields in the export: {sorted(first)}")
print("export fields:", sorted(first), "-- no reference text")


In [ ]:
import time
Path(OUT).mkdir(parents=True, exist_ok=True)

cmd = [sys.executable, "-u", "/kaggle/working/EmoCap/scripts/p4_transfer.py",
       "--data-root", DATA,
       "--runs-root", PREDS,
       "--models", "/kaggle/working/models",
       "--out", f"{OUT}/p4_transfer.json"]
if INCLUDE_PAIRED:
    cmd.append("--include-paired")

t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print("   ", line, end="")
rc = proc.wait()
print(f"\nexit {rc}   [{(time.time()-t0)/60:.1f} min]")
if rc != 0:
    raise RuntimeError(f"p4_transfer.py failed with rc={rc}")


In [ ]:
# The verdict, printed once more so it is visible in the saved notebook without
# re-reading the JSON.
r = json.loads(Path(OUT, "p4_transfer.json").read_text())
v = r["p4_verdict"]
print("judges, own-corpus 5-fold CV:")
for k, j in r["judges"].items():
    print(f"  judge {k}  trained on {j['trained_on']:<12} "
          f"own CV {j.get('own_cv_accuracy')}  sha256 {str(j.get('sha256'))[:16]}")

print("\ncorpus-level transfer (the control):")
for k, val in r["corpus_transfer"].items():
    print(f"  {k:<34} {val}")

print("\ngenerated-caption transfer:")
for arm, row in r["generated"].items():
    print(f"  {arm:<18} n={row['n']:>7,}   " +
          "  ".join(f"{j}-judge {row[f'{j}_judge']:.4f}" for j in ("S", "H")))

print(f"\nP4: S drop {v['S_drop']:+.4f}   H drop {v['H_drop']:+.4f}   "
      f"asymmetry {v['asymmetry_S_minus_H']:+.4f}")
print("   ", "direction holds" if v["p4_direction_holds"]
      else "FALSIFIED -- symmetric or reversed")
print("    No magnitude was registered for P4; direction and size are both reported.")
